In [1]:
import matplotlib
from matplotlib import pyplot as plt
import numpy as np
import scipy.sparse as sparse
import os
import time
import datetime
from matplotlib.colors import LinearSegmentedColormap  

In [2]:
########### FUNCIONES, DERIVADAS Y OTROS ###########
def sparse_DD_neumann(Nx, dx):
    data = np.ones((3, Nx))
    data[1] = -2 * data[1]
    diags = [-1, 0, 1]
    D2 = sparse.spdiags(data, diags, Nx, Nx) / (dx ** 2)
    D2 = sparse.lil_matrix(D2)
    # Condiciones de borde de Neumann: ajustar la primera y última fila
    D2[0, 0] = -1 / (dx ** 2)
    D2[0, 1] = 1 / (dx ** 2)
    D2[-1, -1] = -1 / (dx ** 2)
    D2[-1, -2] = 1 / (dx ** 2)
    return D2.tocsr()

def sparse_DD_periodic(Nx, dx):
    data = np.ones((3, Nx))
    data[1, :] = -2.0
    diags = [-1, 0, 1]
    D2 = sparse.spdiags(data, diags, Nx, Nx, format='lil') / dx ** 2
    D2[0, -1] = 1.0 / dx**2
    D2[-1, 0] = 1.0 / dx**2
    return D2.tocsr()

def Der(D, f):
    d_f = D @ f
    return d_f

def saving_directory(string_parameters, parameter_names):
    save_directory = disc + route + project_name
    for i in range(len(string_parameters)):
        save_directory += "/" + parameter_names[i] + "=" + string_parameters[i]
    if not os.path.exists(save_directory):
        os.makedirs(save_directory)
    return save_directory

In [4]:
########### ECUACIONES DE MOVIMIENTO ###########
def equations_FD(eq, field_slices, t_i, x_grid, y_grid, parameters, operators):
    if eq == 'sine_gordon':
        U_1 = field_slices[0]
        U_2 = field_slices[1]
        DDx = operators[0]
        c = parameters[0]
        gamma = parameters[1]
        alpha = parameters[2]

        ddU = Der(DDx, np.transpose(U_1))

        F = U_2
        G = c ** 2 * ddU - gamma * U_2 + alpha * (U_1 - U_1 ** 3) / 3
        fields = np.array([F, G])

    elif eq == 'PDNLS':

        ############# Dt ψ = - (μ + iν)ψ - iα DDx ψ - iβ|ψ|^2 ψ + γψ* #############

        U_1 = field_slices[0]
        U_2 = field_slices[1]

        alpha = parameters[0]
        beta = parameters[1]
        gamma = parameters[2]
        gamma_real = gamma[0]
        gamma_imag = gamma[1]
        mu = parameters[3]
        nu = parameters[4]

        DD = operators[0]

        ddU_1 = Der(DD, U_1)
        ddU_2 = Der(DD, U_2)

        F = alpha * ddU_2 + (beta * (U_1 ** 2 + U_2 ** 2) + nu + gamma_imag) * U_2 + (gamma_real - mu) * U_1
        G = -alpha * ddU_1 - (beta * (U_1 ** 2 + U_2 ** 2) + nu - gamma_imag) * U_1 - (gamma_real + mu) * U_2

        fields = np.array([F, G])

    elif eq == 'PDNLS_complex':
        U = field_slices[0]

        alpha = parameters[0]
        beta = parameters[1]
        gamma = parameters[2]
        gamma = gamma[0]
        mu = parameters[3]
        nu = parameters[4]

        DD = operators[0]
        ddU = Der(DD, U)

        F = - (mu + 1j * nu) * U - 1j * alpha * ddU - 1j * beta * np.abs(U) * U + gamma * np.conjugate(U)

        fields = np.array([F])
    return fields

In [5]:
########### INTEGRADORES TEMPORALES ###########
def RK4_FD(eq, fields, parameters, grids, dt, Nt, operators, t_rate):
    t_grid = grids[0]  # Time grid
    x_grid = grids[1]  # Spatial grid (x)
    y_grid = grids[2]  # Spatial grid (y) if needed

    fields_history = []
    time_grid = []

    for i in range(Nt - 1):
        t = t_grid[i]
        old_fields = fields.copy()

        k_1 = equations_FD(eq, old_fields, t, x_grid, y_grid, parameters, operators)
        k_2 = equations_FD(eq, old_fields + 0.5 * dt * k_1, t + 0.5 * dt, x_grid, y_grid, parameters, operators)
        k_3 = equations_FD(eq, old_fields + 0.5 * dt * k_2, t + 0.5 * dt, x_grid, y_grid, parameters, operators)
        k_4 = equations_FD(eq, old_fields + dt * k_3, t + dt, x_grid, y_grid, parameters, operators)

        new_fields = old_fields + dt * (k_1 + 2 * k_2 + 2 * k_3 + k_4) / 6
        fields = new_fields

        # Save each t_rate amount of time
        if i % t_rate == 0:
            fields_history.append(fields.copy())
            time_grid.append(t)

    return fields, fields_history, time_grid

In [ ]:
if __name__ == '__main__':

    # Definiendo ubicaciones
    project_name = '/pdnlS'
    disc = 'D:/'                                        # DISCO DE TRABAJO
    route = 'simulation_data/'                          # CARPETA DE TRABAJO
    eq = 'PDNLS'                                        # ECUACION
    t_rate = 100                                        # CADA CUANTAS ITERACIONES GUARDA
    dt = 0.01
    T = 4000
    dx = 0.5




# BARRIDO DE CARTOGRAFÍA
    ies = np.arange(-0.13, -0.2950, -0.005)
    jotas = np.array([0.1])

    [tmin, tmax, dt] = [0, T, dt]
    [xmin, xmax, dx] = [-50, 50, dx]
    t_grid = np.arange(tmin, tmax + dt, dt)
    x_grid = np.arange(xmin, xmax, dx)
    T = tmax
    Nt = t_grid.shape[0]
    Nx = x_grid.shape[0]

    ### Condiciones iniciales de carpeta particular ###

    print("N° of simulations: " + str(len(ies) * len(jotas)))
    for i in ies:
        for j in jotas:
            alpha = 6.524
            beta = 1.0
            mu = 0.1
            nu = i
            gamma_0 = j


            ### CONDICIÓN INICIAL: solitón ###
        
            #  "psi = sqrt(2*delta) * sech(sqrt(delta/alpha)*x) * exp(-i*theta)"
            #  delta = -nu + sqrt(gamma^2 - mu^2)
            U_1_init = np.zeros_like(x_grid)
            U_2_init = np.zeros_like(x_grid)
            x0=-1
            if gamma_0 >= mu:                             
                termino_raiz = np.sqrt(gamma_0**2 - mu**2)
                delta = -nu + termino_raiz
                if delta > 0:                  
                    theta = 0.5 * np.arccos(mu / gamma_0)
                                      
                    A_sol = np.sqrt(2 * delta)              
                    arg_sech = np.sqrt(delta / alpha) * (x_grid-x0) 
                    sech_profile = A_sol * (1.0 / np.cosh(arg_sech))
                    
                    U_1_init = sech_profile * np.cos(theta)
                    U_2_init = sech_profile * -np.sin(theta) 
                    
                else:
                    U_1_init = np.zeros_like(x_grid)
                    U_2_init = np.zeros_like(x_grid)

            alpha_str = f"{alpha:.{4}f}"
            beta_str = f"{beta:.{4}f}"
            gamma_str = f"{gamma_0:.{4}f}"
            mu_str = f"{mu:.{4}f}"
            nu_str = f"{nu:.{4}f}"

            if i == ies[0] and j == jotas[0]:
                print('gamma = ' + gamma_str)
                print('mu = ' + mu_str)
                print('nu = ' + nu_str)
            print('### i = ' + str(i) + ' ###')
            print('### j = ' + str(j) + ' ###')

            ### Empaquetamiento de parametros, campos y derivadas para integración ###
            L = xmax - xmin
            D2 = sparse_DD_neumann(Nx, dx)
            operators = np.array([D2])
            fields_init = [U_1_init, U_2_init]
            grids = [t_grid, x_grid, 0]

     
            ### FORZAMIENTO HETEROGÉNEO  ###
            # gamma(x) = gamma_0 * exp(-x^2 / 2*sigma^2)
            
            sigma = 12.0  # Ancho de la inyección 
            
            # perfil Gaussiano
            gamma_spatial_real = gamma_0 * np.exp(-x_grid**2 / (2 * sigma**2))
            gamma_spatial_img  = np.zeros_like(x_grid) 
            gamma = [gamma_spatial_real, gamma_spatial_img]
            parameters = [alpha, beta, gamma, mu, nu]

            ### INTEGRACIÓN TEMPORAL ###
            now = datetime.datetime.now()
            print('Hora de Inicio: ' + str(now.hour) + ':' + str(now.minute) + ':' + str(now.second))
            time_init = time.time()

            final_fields, fields_history, time_grid = RK4_FD(eq, fields_init, parameters, grids, dt, Nt, operators, t_rate)

            now = datetime.datetime.now()
            print('Hora de Término: ' + str(now.hour) + ':' + str(now.minute) + ':' + str(now.second))
            time_fin = time.time()
            print('TIEMPO TOTAL: ' + f"{(time_fin - time_init):.{3}f}" + ' seg')
            print('##############################')

            #REOBTENIENDO CAMPOS FINALES
            U1_light = np.array(fields_history)[:, 0]
            U2_light = np.array(fields_history)[:, 1]
            U_complex = U1_light + 1j * U2_light
            t_light = time_grid


            modulo_light_1 = np.absolute(U_complex)
            arg_light_1 = np.angle(U_complex)
            arg_light_1 = (2 * np.pi + arg_light_1) * (arg_light_1 < 0) + arg_light_1 * (arg_light_1 > 0)


            params_names = ["alpha", "beta", "gamma", "mu", "nu"]
            str_params = [alpha_str, beta_str, gamma_str, mu_str, nu_str]
            parameters_np = np.array([alpha, beta, gamma_0, mu, nu])
            save_directory = saving_directory(str_params, params_names)

            np.savetxt(save_directory + '/field_real.txt', U1_light, delimiter=',')
            np.savetxt(save_directory + '/field_img.txt', U2_light, delimiter=',')
            np.savetxt(save_directory + '/parameters.txt', parameters_np, delimiter=',')
            np.savetxt(save_directory + '/parameter_names.txt', np.array(params_names), fmt="%s")
            np.savetxt(save_directory + '/X.txt', x_grid, delimiter=',')
            np.savetxt(save_directory + '/T.txt', t_light, delimiter=',')



            pcm = plt.pcolormesh(x_grid, t_light, modulo_light_1, cmap='viridis', shading='auto')
            cbar = plt.colorbar(pcm, shrink=1)
            cbar.set_label('$|A|$', rotation=0, size=20, labelpad=-27, y=1.1)
            plt.xlim([x_grid[0], x_grid[-1]])
            plt.xlabel('$x$', size='20')
            plt.ylabel('$t$', size='20')
            plt.grid(linestyle='--', alpha=0.5)
            plt.savefig(save_directory + '/module_spacetime.png', dpi=300)
            plt.close()

            pcm = plt.pcolormesh(x_grid, t_light, arg_light_1, cmap='viridis', shading='auto')
            cbar = plt.colorbar(pcm, shrink=1)
            cbar.set_label('$arg(A)$', rotation=0, size=20, labelpad=-20, y=1.1)
            plt.xlim([x_grid[0], x_grid[-1]])
            plt.xlabel('$x$', size='20')
            plt.ylabel('$t$', size='20')
            plt.grid(linestyle='--', alpha=0.5)
            plt.savefig(save_directory + '/arg_spacetime.png', dpi=300)
            plt.close()

            pcm = plt.pcolormesh(x_grid, t_light, U1_light, cmap='viridis', vmin=-np.amax(U1_light), vmax=np.amax(U1_light), shading='auto')
            cbar = plt.colorbar(pcm, shrink=1)
            cbar.set_label('$A_R(x, t)$', rotation=0, size=20, labelpad=-27, y=1.1)
            plt.xlim([x_grid[0], x_grid[-1]])
            plt.xlabel('$x$', size='20')
            plt.ylabel('$t$', size='20')
            plt.grid(linestyle='--', alpha=0.5)
            plt.savefig(save_directory + '/real_spacetime.png', dpi=300)
            plt.close()

            pcm = plt.pcolormesh(x_grid, t_light, U2_light, cmap='viridis', vmin=-np.amax(U1_light), vmax=np.amax(U1_light), shading='auto')
            cbar = plt.colorbar(pcm, shrink=1)
            cbar.set_label('$A_I(x, t)$', rotation=0, size=20, labelpad=-27, y=1.1)
            plt.xlim([x_grid[0], x_grid[-1]])
            plt.xlabel('$x$', size='20')
            plt.ylabel('$t$', size='20')
            plt.grid(linestyle='--', alpha=0.5)
            plt.savefig(save_directory + '/img_spacetime.png', dpi=300)
            plt.close()
    print('##########  FIN  ##########')

N° of simulations: 33
gamma = 0.1000
mu = 0.1000
nu = -0.1300
### i = -0.13 ###
### j = 0.1 ###
Hora de Inicio: 22:46:14
Hora de Término: 22:47:20
TIEMPO TOTAL: 65.713 seg
##############################
### i = -0.135 ###
### j = 0.1 ###
Hora de Inicio: 22:47:26
Hora de Término: 22:48:29
TIEMPO TOTAL: 63.116 seg
##############################
### i = -0.14 ###
### j = 0.1 ###
Hora de Inicio: 22:48:36
Hora de Término: 22:49:39
TIEMPO TOTAL: 62.761 seg
##############################
### i = -0.14500000000000002 ###
### j = 0.1 ###
Hora de Inicio: 22:49:45
Hora de Término: 22:50:48
TIEMPO TOTAL: 62.870 seg
##############################
### i = -0.15000000000000002 ###
### j = 0.1 ###
Hora de Inicio: 22:50:54
Hora de Término: 22:51:57
TIEMPO TOTAL: 62.893 seg
##############################
### i = -0.15500000000000003 ###
### j = 0.1 ###
Hora de Inicio: 22:52:3
Hora de Término: 22:53:6
TIEMPO TOTAL: 63.099 seg
##############################
### i = -0.16000000000000003 ###
### j = 0.1 ###

In [ ]:
import os
import gc
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from scipy import integrate

# ==========================================
#  CONFIGURACIÓN DE GRÁFICOS
# ==========================================
plt.rcParams.update({
    "font.family": "serif",        
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.linewidth": 1.5,        
    "axes.grid": True,
    "grid.alpha": 0.5,
    "grid.linestyle": "--",
    "legend.fontsize": 12,
    "legend.framealpha": 0.9
})

# ==========================================
#  VARIABLES
# ==========================================
base_path = r'D:\simulation_data\pdnlS\alpha=6.5240\beta=1.0000'

# Tolerancias
TOL_STD_POS = 0.05
TOL_STD_AMP = 0.015
TOL_CENTER  = 0.5
CUT_OFF     = 0.2

results_nu = []
results_gamma = []
results_state = []

def classify_dynamics(t, center, amplitude):
    """Clasifica el estado."""
    n = len(t)
    start = int(n * (1 - CUT_OFF))
    c_tail = center[start:]
    a_tail = amplitude[start:]
    
    if np.mean(a_tail) < 0.1: return "Decay"
    
    mean_pos = np.mean(c_tail)
    std_amp = np.std(a_tail)
    
    if len(c_tail) > 1:
        coeffs = np.polyfit(np.arange(len(c_tail)), c_tail, 1)
        slope = coeffs[0]
    else: slope = 0
    
    is_breathing = std_amp > TOL_STD_AMP
    is_drifting  = abs(mean_pos) > TOL_CENTER or abs(slope) > 0.001
    
    if is_drifting and is_breathing: return "Drifting Breather"
    elif is_drifting: return "Stationary Drifted"
    elif is_breathing: return "Stationary Breather"
    else: return "Stationary Soliton"

# ==========================================
# 2. BUCLE PRINCIPAL
# ==========================================
print(f"Generando gráficos comparativos (T=2000 a 4000)...")

if os.path.exists(base_path):
    gammas_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d)) and 'gamma=' in d]
    gammas_dirs.sort()
    
    for g_dir in gammas_dirs:
        try:
            gamma_val = float(g_dir.split('=')[1])
            path_gamma = os.path.join(base_path, g_dir)
            
            #  figura comparativa
            fig_comp, ax_comp = plt.subplots(1, 3, figsize=(18, 5))
            fig_comp.suptitle(f"Comparación $\gamma = {gamma_val:.4f}$ (t=2000-4000)", fontsize=18)
            has_plots = False
            
            mus_dirs = [d for d in os.listdir(path_gamma) if os.path.isdir(os.path.join(path_gamma, d)) and 'mu=' in d]
            
            for m_dir in mus_dirs:
                path_mu = os.path.join(path_gamma, m_dir)
                nus_dirs = [d for d in os.listdir(path_mu) if os.path.isdir(os.path.join(path_mu, d)) and 'nu=' in d]
                nus_dirs.sort(key=lambda x: float(x.split('=')[1]), reverse=False) 
                
                # --- MAPA DE COLORES ---
                valores_nu_reales = []
                for n_dir in nus_dirs:
                    try: valores_nu_reales.append(float(n_dir.split('=')[1]))
                    except: pass
                
                if valores_nu_reales:
                    norm = Normalize(vmin=min(valores_nu_reales), vmax=max(valores_nu_reales))
                    cmap = plt.get_cmap('viridis')
                
                for n_dir in nus_dirs:
                    try:
                        nu_val = float(n_dir.split('=')[1])
                        sim_path = os.path.join(path_mu, n_dir)
                        
                        try:
                            U_real = np.loadtxt(os.path.join(sim_path, 'field_real.txt'), delimiter=',')
                            U_imag = np.loadtxt(os.path.join(sim_path, 'field_img.txt'), delimiter=',')
                            x_grid = np.loadtxt(os.path.join(sim_path, 'X.txt'), delimiter=',')
                            t_grid = np.loadtxt(os.path.join(sim_path, 'T.txt'), delimiter=',')
                        except: continue

                        has_plots = True
                        
                        # Cálculos básicos
                        U = U_real + 1j * U_imag
                        U_mod = np.abs(U)
                        N = integrate.simpson(U_mod ** 2, x_grid, axis=1)
                        
                        with np.errstate(divide='ignore', invalid='ignore'):
                            center = integrate.simpson(x_grid * U_mod ** 2, x_grid, axis=1) / N
                            center = np.nan_to_num(center)
                            
                        width = []
                        for i in range(len(t_grid)):
                            if N[i] > 1e-6:
                                w = integrate.simpson((x_grid - center[i])**2 * U_mod[i,:]**2, x_grid) / N[i]
                            else: w = 0
                            width.append(w)
                        width = np.array(width)
                        
                        Max_Amp = np.max(U_mod, axis=1)
                        state = classify_dynamics(t_grid, center, Max_Amp)
                        
                        results_nu.append(nu_val)
                        results_gamma.append(gamma_val)
                        results_state.append(state)
                        
                        print(f"  G={gamma_val:.3f} | Nu={nu_val:.3f} -> {state}")
                        
                        # ===============================================
                        # 1. GRÁFICO INDIVIDUAL
                        # ===============================================
                        mask_ind = t_grid >= 2000 
                        if np.sum(mask_ind) == 0: mask_ind = t_grid >= 0 
                        
                        t_ind = t_grid[mask_ind][::5]
                        c_ind = center[mask_ind][::5]
                        N_ind = N[mask_ind][::5]
                        w_ind = width[mask_ind][::5]

                        fig_ind, ax_ind = plt.subplots(1, 3, figsize=(14, 4))
                        ax_ind[0].plot(t_ind, c_ind, color='#1f77b4', linewidth=2)
                        ax_ind[0].set_title(r"Posición $x_{cm}$")
                        ax_ind[1].plot(t_ind, N_ind, color='#d62728', linewidth=2)
                        ax_ind[1].set_title("Energía")
                        ax_ind[2].plot(t_ind, w_ind, color='#2ca02c', linewidth=2)
                        ax_ind[2].set_title("Ancho")
                        
                        for ax in ax_ind: ax.set_xlabel(r"$t$")
                        plt.tight_layout()
                        plt.savefig(os.path.join(sim_path, 'individual_plot.png'), dpi=200, bbox_inches='tight')
                        plt.close(fig_ind) 
                        
                        mask_comp = t_grid >= 2000
                        if np.sum(mask_comp) == 0: mask_comp = t_grid >= 0
                        t_comp = t_grid[mask_comp][::5]
                        c_comp = center[mask_comp][::5]
                        N_comp = N[mask_comp][::5]
                        w_comp = width[mask_comp][::5]

                        color_linea = cmap(norm(nu_val))
                        
                        ax_comp[0].plot(t_comp, c_comp, color=color_linea, alpha=0.7)
                        ax_comp[1].plot(t_comp, N_comp, color=color_linea, alpha=0.7)
                        ax_comp[2].plot(t_comp, w_comp, color=color_linea, alpha=0.7)

                    except Exception as e:
                        print(f"Error procesando {n_dir}: {e}")
                        
                    finally:
                        try: del U_real, U_imag, U, U_mod, x_grid, t_grid, center, N, width
                        except NameError: pass
                        gc.collect()

            if has_plots:
                ax_comp[0].set_ylabel(r"$x_s$"); ax_comp[0].set_xlabel(r"$t$")
                ax_comp[1].set_ylabel(r"$N$"); ax_comp[1].set_xlabel(r"$t$")
                ax_comp[2].set_ylabel(r"$\delta$"); ax_comp[2].set_xlabel(r"$t$")
                plt.tight_layout()
                plt.subplots_adjust(right=0.9) 
                cax = fig_comp.add_axes([0.92, 0.15, 0.015, 0.7]) 
                
                sm = cm.ScalarMappable(cmap=cmap, norm=norm)
                sm.set_array([]) 
                cbar = fig_comp.colorbar(sm, cax=cax) 
                cbar.set_label(r'Detuning $\nu$', size=16)
                
                plt.savefig(os.path.join(path_gamma, f'COMPARACION_gamma_{gamma_val:.4f}.png'), dpi=250)
                plt.close(fig_comp)
                plt.close('all')
                gc.collect()
                
        except Exception as e:
            print(f"Error gamma {g_dir}: {e}")

else:
    print("Ruta base no existe")

# ==========================================
# 3. MAPA DE FASES
# ==========================================
if len(results_state) > 0:
    plt.figure(figsize=(10, 8))
    colors_map = {
        "Decay": "black",
        "Stationary Soliton": "green",
        "Stationary Breather": "blue",
        "Stationary Drifted": "orange",
        "Drifting Breather": "magenta"
    }
    
    arr_nu = np.array(results_nu)
    arr_gamma = np.array(results_gamma)
    arr_state = np.array(results_state)
    
    for state in np.unique(arr_state):
        mask = (arr_state == state)
        plt.scatter(
            arr_nu[mask], arr_gamma[mask], 
            c=colors_map.get(state, 'gray'), label=state, 
            s=25, edgecolors='k', linewidths=0.4
        )

    plt.title("Diagrama de Fases Completo")
    plt.xlabel(r"Detuning $\nu$")
    plt.ylabel(r"Forzamiento $\gamma$")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', framealpha=1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(base_path, 'Mapa_Fases_Completo.png'), dpi=300, bbox_inches='tight')
    
    print("¡Proceso finalizado! Los comparativos ahora muestran T=2000 a 4000.")
else:
    print("No se encontraron datos.")

<>:86: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<>:86: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
C:\Users\panch\AppData\Local\Temp\ipykernel_17636\354429576.py:86: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
  fig_comp.suptitle(f"Comparación $\gamma = {gamma_val:.4f}$ (t=2000-4000)", fontsize=18)


Generando gráficos comparativos (T=2000 a 4000)...
  G=0.000 | Nu=-0.400 -> Decay
  G=0.000 | Nu=-0.395 -> Decay
  G=0.000 | Nu=-0.390 -> Decay
  G=0.000 | Nu=-0.385 -> Decay
  G=0.000 | Nu=-0.380 -> Decay
  G=0.000 | Nu=-0.375 -> Decay
  G=0.000 | Nu=-0.370 -> Decay
  G=0.000 | Nu=-0.365 -> Decay
  G=0.000 | Nu=-0.360 -> Decay
  G=0.000 | Nu=-0.355 -> Decay
  G=0.000 | Nu=-0.350 -> Decay
  G=0.000 | Nu=-0.345 -> Decay
  G=0.000 | Nu=-0.340 -> Decay
  G=0.000 | Nu=-0.335 -> Decay
  G=0.000 | Nu=-0.330 -> Decay
  G=0.000 | Nu=-0.325 -> Decay
  G=0.000 | Nu=-0.320 -> Decay
  G=0.000 | Nu=-0.315 -> Decay
  G=0.000 | Nu=-0.310 -> Decay
  G=0.000 | Nu=-0.305 -> Decay
  G=0.000 | Nu=-0.300 -> Decay
  G=0.000 | Nu=-0.295 -> Decay
  G=0.000 | Nu=-0.290 -> Decay
  G=0.000 | Nu=-0.285 -> Decay
  G=0.000 | Nu=-0.280 -> Decay
  G=0.000 | Nu=-0.275 -> Decay
  G=0.000 | Nu=-0.270 -> Decay
  G=0.000 | Nu=-0.265 -> Decay
  G=0.000 | Nu=-0.260 -> Decay
  G=0.000 | Nu=-0.255 -> Decay
  G=0.000 | Nu=-0.2

In [ ]:
import os
import gc
import numpy as np
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
from scipy import integrate

# ==========================================
# 0. ESTILO CIENTÍFICO
# ==========================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "lines.linewidth": 1.5,
    "axes.grid": True,
    "grid.alpha": 0.5,
    "grid.linestyle": "--",
    "legend.fontsize": 11,
    "legend.framealpha": 0.9
})

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
base_path = r'D:\simulation_data\pdnlS\alpha=6.5240\beta=1.0000'
gamma_fijo = 0.2000  

PORCENTAJE_FINAL = 0.3 # último 30% de la simulación
UMBRAL_VIDA = 0.1      

list_nu = []
list_x_mean = []
list_x_min = []
list_x_max = []
list_status = [] 

print(f"--- Generando Bifurcación de POSICIÓN (Tenedor) Gamma={gamma_fijo} ---")
cadena_gamma = f"gamma={gamma_fijo:.4f}"

# ==========================================
# 2. PROCESAMIENTO
# ==========================================
if os.path.exists(base_path):
    for root, dirs, files in os.walk(base_path):
        if cadena_gamma in root or cadena_gamma in os.path.basename(root):
            if 'field_real.txt' in files and 'X.txt' in files:
                try:
                    nombre = os.path.basename(root)
                    if 'nu=' in nombre:
                     
                        nu_val = float(nombre.split('nu=')[1].split('_')[0])
                    else:
                     
                        for p in root.split(os.sep): 
                            if 'nu=' in p: nu_val = float(p.split('=')[1]); break
                    
                    # ---  CARGAR DATOS ---
                    U_real = np.loadtxt(os.path.join(root, 'field_real.txt'), delimiter=',')
                    U_imag = np.loadtxt(os.path.join(root, 'field_img.txt'), delimiter=',')
                    x_grid = np.loadtxt(os.path.join(root, 'X.txt'), delimiter=',')
                    
                    # ---  CALCULAR CENTRO DE MASA ---
                    U_mod2 = U_real**2 + U_imag**2
                    Norm = integrate.simpson(U_mod2, x_grid, axis=1)
                    
                    if Norm[-1] < UMBRAL_VIDA:
                        list_nu.append(nu_val)
                        list_x_mean.append(0)
                        list_x_min.append(0)
                        list_x_max.append(0)
                        list_status.append('Decay')
                        print(f"  Nu={nu_val:.3f} -> Decay")
                        continue

                    # Cálculo de Posición 
                    with np.errstate(divide='ignore', invalid='ignore'):
                        Center = integrate.simpson(x_grid * U_mod2, x_grid, axis=1) / Norm
                        Center = np.nan_to_num(Center)
                    
                
                    n_puntos = len(Center)
                    n_corte = int(n_puntos * (1 - PORCENTAJE_FINAL))
                    
               
                    x_tail = Center[n_corte:]
                    
                    mean_x = np.mean(x_tail)
                    min_x = np.min(x_tail)
                    max_x = np.max(x_tail)
                    
                    list_nu.append(nu_val)
                    list_x_mean.append(mean_x)
                    list_x_min.append(min_x)
                    list_x_max.append(max_x)
                    list_status.append('Alive')
                    
                    print(f"  Nu={nu_val:.3f} | Pos={mean_x:.2f}")

                except Exception as e:
                    print(f"Error procesando {root}: {e}")
                
                finally:
                    # --- LIMPIEZA DE MEMORIA VITAL ---
                    try: del U_real, U_imag, U_mod2, Center, Norm
                    except: pass
                    gc.collect()

# ==========================================
# 3. GRAFICAR BIFURCACIÓN
# ==========================================
if len(list_nu) > 0:
    zipped = sorted(zip(list_nu, list_x_mean, list_x_min, list_x_max, list_status))
    nu_arr, xm_arr, xmin_arr, xmax_arr, st_arr = zip(*zipped)
    
    nu_arr = np.array(nu_arr)
    xm_arr = np.array(xm_arr)
    xmin_arr = np.array(xmin_arr)
    xmax_arr = np.array(xmax_arr)
    st_arr = np.array(st_arr)
    
    plt.figure(figsize=(12, 7))
    
    mask_decay = (st_arr == 'Decay')
    if np.any(mask_decay):
        plt.scatter(nu_arr[mask_decay], xm_arr[mask_decay], color='gray', s=30, label='Decay (Muerto)', zorder=5, alpha=0.7)


    mask_alive = (st_arr == 'Alive')
    if np.any(mask_alive):
        plt.plot(nu_arr[mask_alive], xm_arr[mask_alive], 'o-', color='blue', label='Rama Simulada ($x_0=-1$)', markersize=5)
        plt.fill_between(nu_arr[mask_alive], xmin_arr[mask_alive], xmax_arr[mask_alive], color='blue', alpha=0.2)
        plt.plot(nu_arr[mask_alive], -xm_arr[mask_alive], 'o-', color='red', label='Rama Espejo (Simetría)', markersize=5)
        plt.fill_between(nu_arr[mask_alive], -xmin_arr[mask_alive], -xmax_arr[mask_alive], color='red', alpha=0.2)
    plt.axhline(0, color='gray', linestyle='--', alpha=0.5)

    plt.title(rf"Diagrama de Bifurcación: Posición ($\gamma = {gamma_fijo}$)")
    plt.xlabel(r"Detuning $\nu$")
    plt.ylabel(r"Posición Final $x$")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Nombre de archivo dinámico
    nombre_salida = f'Bifurcacion_POSICION_Gamma_{gamma_fijo:.4f}.png'
    path_salida = os.path.join(base_path, nombre_salida)
    
    plt.savefig(path_salida, dpi=300)
    print(f"\n¡Gráfico guardado exitosamente en:\n{path_salida}")
    
else:
    print("No se encontraron datos para graficar.")

--- Generando Bifurcación de POSICIÓN (Tenedor) Gamma=0.2 ---
  Nu=-0.000 | Pos=-0.00
  Nu=-0.005 | Pos=0.00
  Nu=-0.010 | Pos=0.00
  Nu=-0.015 | Pos=0.00
  Nu=-0.020 | Pos=0.00
  Nu=-0.025 | Pos=0.00
  Nu=-0.030 | Pos=0.00
  Nu=-0.035 | Pos=0.00
  Nu=-0.040 | Pos=0.00
  Nu=-0.045 | Pos=0.00
  Nu=-0.050 | Pos=0.00
  Nu=-0.055 | Pos=0.00
  Nu=-0.060 | Pos=0.00
  Nu=-0.065 | Pos=0.00
  Nu=-0.070 | Pos=0.00
  Nu=-0.075 | Pos=0.00
  Nu=-0.080 | Pos=0.00
  Nu=-0.085 | Pos=0.00
  Nu=-0.090 | Pos=0.00
  Nu=-0.095 | Pos=0.00
  Nu=-0.100 | Pos=0.00
  Nu=-0.105 | Pos=0.00
  Nu=-0.110 | Pos=0.00
  Nu=-0.115 | Pos=0.00
  Nu=-0.120 | Pos=0.00
  Nu=-0.125 | Pos=0.00
  Nu=-0.130 | Pos=0.00
  Nu=-0.135 | Pos=0.00
  Nu=-0.140 | Pos=0.00
  Nu=-0.145 | Pos=-0.00
  Nu=-0.150 | Pos=-0.00
  Nu=-0.155 | Pos=-0.00
  Nu=-0.160 | Pos=-0.00
  Nu=-0.165 | Pos=-0.00
  Nu=-0.170 | Pos=-0.01
  Nu=-0.175 | Pos=-0.13
  Nu=-0.180 | Pos=-1.68
  Nu=-0.185 | Pos=-4.04
  Nu=-0.190 | Pos=-5.34
  Nu=-0.195 | Pos=-6.34
  Nu=-

In [ ]:
import os
import gc
import numpy as np
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq
from scipy.signal import detrend
from scipy import integrate

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
ruta_base = r'D:\simulation_data\pdnlS\alpha=6.5240\beta=1.0000'
gamma_objetivo = 0.2000  
PORCENTAJE_FINAL = 0.30  # Analizar solo el último 30%


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.grid": True,
    "grid.linestyle": "--",
    "lines.linewidth": 1.5
})


datos_nu = []
freqs_amplitud = []
freqs_posicion = []

print(f"--- ANÁLISIS DE FRECUENCIAS (AMPLITUD Y POSICIÓN) ---")
print(f"Gamma: {gamma_objetivo} | Último {PORCENTAJE_FINAL*100:.0f}%")

# ==========================================
# 2. RECOLECCIÓN Y ORDENAMIENTO
# ==========================================
carpetas_procesar = []
cadena_gamma = f"gamma={gamma_objetivo:.4f}"

if os.path.exists(ruta_base):
    for root, dirs, files in os.walk(ruta_base):
        if cadena_gamma in root or cadena_gamma in os.path.basename(root):
            if 'field_real.txt' in files and 'X.txt' in files and 'T.txt' in files:
                try:
                    nombre = os.path.basename(root)
                    if 'nu=' in nombre:
                        val = float(nombre.split('nu=')[1].split('_')[0])
                        carpetas_procesar.append((val, root))
                except: continue

carpetas_procesar.sort(key=lambda x: x[0])
print(f"Se encontraron {len(carpetas_procesar)} simulaciones.")

# ==========================================
# 3. PROCESAMIENTO
# ==========================================
for nu_val, carpeta in carpetas_procesar:
    try:

        t = np.loadtxt(os.path.join(carpeta, 'T.txt'), delimiter=',')
        N_total = len(t)
        n_keep = int(N_total * PORCENTAJE_FINAL)
        t_cut = t[-n_keep:]
        dt = t_cut[1] - t_cut[0]
        
        print(f"Nu={nu_val:.4f} | T: {t_cut[0]:.1f}->{t_cut[-1]:.1f}", end=" | ")

        x_grid = np.loadtxt(os.path.join(carpeta, 'X.txt'), delimiter=',')
    
        u_real = np.loadtxt(os.path.join(carpeta, 'field_real.txt'), delimiter=',')[-n_keep:, :]
        u_imag = np.loadtxt(os.path.join(carpeta, 'field_img.txt'), delimiter=',')[-n_keep:, :]
        
        # C. CÁLCULO 1: FRECUENCIA DE AMPLITUD
        # A(t) = max(|psi|) en el espacio
        Campo_Mod = np.abs(u_real + 1j * u_imag)
        Amplitud_t = np.max(Campo_Mod, axis=1)
        
        # FFT Amplitud
        amp_detrend = detrend(Amplitud_t, type='linear')
        yf_a = rfft(amp_detrend)
        xf_a = rfftfreq(len(amp_detrend), dt)
        pot_a = np.abs(yf_a)
        
        # D. CÁLCULO 2: FRECUENCIA DE POSICIÓN
        # Pos(t) = Integral(x * |psi|^2) / N
        Densidad = Campo_Mod**2
        Norma = integrate.simpson(Densidad, x_grid, axis=1)
        
        if np.mean(Norma) < 0.1:
            print("Decay (Muerto)")
            continue

        with np.errstate(divide='ignore', invalid='ignore'):
            Momento = integrate.simpson(x_grid * Densidad, x_grid, axis=1)
            Posicion_t = Momento / Norma
            Posicion_t = np.nan_to_num(Posicion_t)

        # FFT Posición
        pos_detrend = detrend(Posicion_t, type='linear')
        yf_p = rfft(pos_detrend)
        xf_p = rfftfreq(len(pos_detrend), dt)
        pot_p = np.abs(yf_p)

        # E. EXTRAER PICOS
        freq_amp = 0.0
        freq_pos = 0.0
        
        # Pico Amplitud
        mask_a = xf_a > 0.005
        if np.any(mask_a):
            idx = np.argmax(pot_a[mask_a])
            if pot_a[mask_a][idx] > 0.01: freq_amp = xf_a[mask_a][idx]

        # Pico Posición
        mask_p = xf_p > 0.005
        if np.any(mask_p):
            idx = np.argmax(pot_p[mask_p])
            if pot_p[mask_p][idx] > 0.01: freq_pos = xf_p[mask_p][idx]
            
        print(f"f_Amp={freq_amp:.3f} | f_Pos={freq_pos:.3f}")
        
        datos_nu.append(nu_val)
        freqs_amplitud.append(freq_amp)
        freqs_posicion.append(freq_pos)

    except Exception as e:
        print(f"Error: {e}")
    finally:
        try: del u_real, u_imag, Campo_Mod, Densidad, t, x_grid
        except: pass
        gc.collect()

# ==========================================
# 4. GRAFICAR (DOS IMÁGENES SEPARADAS)
# ==========================================
if len(datos_nu) > 0:
    
    # Ordenar datos para graficar limpio
    zipped = sorted(zip(datos_nu, freqs_amplitud, freqs_posicion))
    nu_arr, fa_arr, fp_arr = zip(*zipped)
    
    # --- GRÁFICO 1: FRECUENCIA DE AMPLITUD ---
    plt.figure(figsize=(10, 6))
    plt.plot(nu_arr, fa_arr, color='darkmagenta', linewidth=2, zorder=1)
    plt.scatter(nu_arr, fa_arr, color='darkmagenta', s=60, edgecolors='white', linewidths=1.5, zorder=2)
    plt.title(rf"Frecuencia de Amplitud (Último {PORCENTAJE_FINAL*100:.0f}%) $\gamma={gamma_objetivo}$")
    plt.xlabel(r"Detuning $\nu$")
    plt.ylabel(r"Frecuencia $f_A$ (Hz)")
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_base, f'Freq_AMPLITUD_G{gamma_objetivo:.4f}.png'), dpi=300)
    plt.close()

    # --- GRÁFICO 2: FRECUENCIA DE POSICIÓN ---
    plt.figure(figsize=(10, 6))
    plt.plot(nu_arr, fp_arr, color='darkorange', linewidth=2, zorder=1)
    plt.scatter(nu_arr, fp_arr, color='darkorange', s=60, edgecolors='white', linewidths=1.5, zorder=2)
    plt.title(rf"Frecuencia de Posición (Último {PORCENTAJE_FINAL*100:.0f}%) $\gamma={gamma_objetivo}$")
    plt.xlabel(r"Detuning $\nu$")
    plt.ylabel(r"Frecuencia $f_x$ (Hz)")
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_base, f'Freq_POSICION_G{gamma_objetivo:.4f}.png'), dpi=300)
    plt.close()

    print("\n¡Proceso Terminado! Se han guardado las dos imágenes.")
else:
    print("No se encontraron datos válidos.")

--- ANÁLISIS DE FRECUENCIAS (AMPLITUD Y POSICIÓN) ---
Gamma: 0.2 | Último 30%
Se encontraron 81 simulaciones.
Nu=-0.4000 | T: 2800.0->3999.0 | Decay (Muerto)
Nu=-0.3950 | T: 2800.0->3999.0 | Decay (Muerto)
Nu=-0.3900 | T: 2800.0->3999.0 | Decay (Muerto)
Nu=-0.3850 | T: 2800.0->3999.0 | f_Amp=0.068 | f_Pos=0.007
Nu=-0.3800 | T: 2800.0->3999.0 | f_Amp=0.068 | f_Pos=0.028
Nu=-0.3750 | T: 2800.0->3999.0 | f_Amp=0.068 | f_Pos=0.068
Nu=-0.3700 | T: 2800.0->3999.0 | f_Amp=0.068 | f_Pos=0.275
Nu=-0.3650 | T: 2800.0->3999.0 | f_Amp=0.069 | f_Pos=0.069
Nu=-0.3600 | T: 2800.0->3999.0 | f_Amp=0.069 | f_Pos=0.069
Nu=-0.3550 | T: 2800.0->3999.0 | f_Amp=0.070 | f_Pos=0.070
Nu=-0.3500 | T: 2800.0->3999.0 | f_Amp=0.072 | f_Pos=0.072
Nu=-0.3450 | T: 2800.0->3999.0 | f_Amp=0.088 | f_Pos=0.088
Nu=-0.3400 | T: 2800.0->3999.0 | f_Amp=0.091 | f_Pos=0.091
Nu=-0.3350 | T: 2800.0->3999.0 | f_Amp=0.092 | f_Pos=0.092
Nu=-0.3300 | T: 2800.0->3999.0 | f_Amp=0.088 | f_Pos=0.088
Nu=-0.3250 | T: 2800.0->3999.0 | f_Amp